<a href="https://colab.research.google.com/github/iesataulfoargentasor/DocIA-Plus/blob/main/laboratorio/colab/02_chromadb_paso_a_paso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cuaderno de la sesión 1. ChromaDB con el modelo

En el cuaderno anterior el vector vivía en una variable y se perdía al cerrar la sesión. Una base de datos lo guarda para poder buscarlo después.

Este cuaderno no es el laboratorio de vectores escritos a mano. Aquel viene después, en la carpeta `laboratorio/`, y no descarga un modelo.

ChromaDB es la base que usaremos en DocIA+. Aquí trabaja dentro de Colab. Primero en memoria, y al final en una carpeta.

Cada registro guarda cuatro cosas, y cada una tiene un motivo:

- `ids`: identificador único, para actualizar o borrar ese registro y no otro
- `documents`: el texto del trozo, porque el vector no se puede leer ni citar
- `embeddings`: la lista de números, calculada por nosotros con el mismo modelo de antes
- `metadatas`: datos para filtrar, por ejemplo la categoría. No forman parte del vector

Si solo pasáis el texto y no el vector, ChromaDB llama a un modelo que no veis. Puede no ser el de 384 números. En este cuaderno no se hace así.

Las frases siguen siendo de ejercicio.


## Instalar y cargar el mismo modelo

Hace falta la librería de la base (`chromadb`) y otra vez el modelo del cuaderno anterior.

Se imprime la dimensión. Tiene que salir 384. Si no sale 384, este cuaderno y el anterior no están en el mismo mapa y las distancias no significan nada.


In [1]:
!pip install chromadb sentence-transformers -q

from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")
print("Modelo listo. Dimensiones:", modelo.get_sentence_embedding_dimension())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently tak

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo listo. Dimensiones: 384


/tmp/ipykernel_2688/2757363350.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Modelo listo. Dimensiones:", modelo.get_sentence_embedding_dimension())


## Crear la colección

En SQL, antes de insertar filas se crea la tabla. Aquí se crea la colección: el sitio donde vivirán los registros.

`EphemeralClient` la guarda solo en memoria. Al cerrar el cuaderno desaparece. Más adelante veréis la que se escribe en disco.

`hnsw:space = cosine` le dice cómo medir. Hoy basta con esto: en esta colección, una distancia más pequeña significa más parecido. El coseno se calcula a mano en el tema 3.

Si la colección ya existía de una ejecución anterior, se borra y se crea vacía. Así no se mezclan pruebas viejas.


In [2]:
import chromadb

cliente = chromadb.EphemeralClient()
existentes = [c.name for c in cliente.list_collections()]
if "documentacion_ies" in existentes:
    cliente.delete_collection("documentacion_ies")

coleccion = cliente.create_collection(
    name="documentacion_ies",
    metadata={"hnsw:space": "cosine"},
)
print("Coleccion creada:", coleccion.name)


Coleccion creada: documentacion_ies


## Guardar cinco frases

Este es el paso de cargar la base. Cada frase ya es un trozo, así que no hay que partirla.

Primero `encode` calcula las cinco listas con el modelo que acabamos de cargar. Después `add` las guarda. El orden importa: el vector que entra es el nuestro. La base no lo inventa.

La categoría es un metadato. Sirve para filtrar, como un `WHERE`. No cambia los números del vector.

Usamos los mismos códigos que el proyecto: `g4` oferta educativa, `g5` horarios y actividades del centro, `g3` planes. El texto de la frase no es la categoría. «Secretaría» puede aparecer en el texto y el código seguir siendo `g4`.


In [3]:
frases = [
    "Procedimiento de formalizacion de matricula en el curso de especializacion.",
    "Para matricularse hay que presentar la solicitud en secretaria.",
    "El horario de la cafeteria del centro es de 8:00 a 14:00.",
    "Las pruebas de acceso a ciclos formativos se realizan en junio.",
    "El plan de orientacion ayuda al alumnado a elegir estudios.",
]
categorias = [
    {"categoria": "g4", "fuente": "ejemplo-oferta"},
    {"categoria": "g4", "fuente": "ejemplo-acceso"},
    {"categoria": "g5", "fuente": "ejemplo-horarios"},
    {"categoria": "g4", "fuente": "ejemplo-acceso"},
    {"categoria": "g3", "fuente": "ejemplo-orientacion"},
]
ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]
vectores = modelo.encode(frases).tolist()

coleccion.add(
    ids=ids,
    documents=frases,
    embeddings=vectores,
    metadatas=categorias,
)
print("Registros guardados:", coleccion.count())


Registros guardados: 5


## Mirar el vector que quedó dentro

Antes de preguntar, se comprueba que la base ha guardado lo que le hemos dado.

Pedimos el registro `doc1` con su texto y su vector. La longitud tiene que ser 384, la misma del cuaderno anterior. Si la base hubiera llamado a otro modelo a escondidas, esta comprobación fallaría.


In [4]:
interno = coleccion.get(ids=["doc1"], include=["embeddings", "documents"])
texto = interno["documents"][0]
vector = interno["embeddings"][0]
print("Texto:", texto)
print("Dimensiones:", len(vector))
print("Diez primeros numeros:", list(vector[:10]))


Texto: Procedimiento de formalizacion de matricula en el curso de especializacion.
Dimensiones: 384
Diez primeros numeros: [np.float64(0.022833483293652534), np.float64(0.03755030035972595), np.float64(-0.05477689206600189), np.float64(-0.056928399950265884), np.float64(-0.05337606370449066), np.float64(-0.030209507793188095), np.float64(-0.03326735645532608), np.float64(0.004482160788029432), np.float64(0.04146776348352432), np.float64(0.02636851742863655)]


## Preguntar con otras palabras

La pregunta también es un dato. Hay que convertirla en vector con el mismo modelo y comparar esa lista con las que están guardadas. `query_embeddings` hace la comparación. `n_results=2` pide solo los dos vecinos más cercanos.

La frase no contiene «formalizacion» ni «matricula». Un `LIKE` no encontraría la primera frase. La base vectorial sí puede, porque no busca las letras: busca el punto más cercano.

La distancia que imprime ChromaDB en esta colección es 1 menos el coseno: más pequeña cuanto más se parecen, y 0 si los vectores son iguales. No es la resta de listas del cuaderno anterior. Un 0,002 aquí puede ser casi el mismo texto. No lo compares con el 0,8 de aquella cuenta.


In [5]:
pregunta = "Como me inscribo en el curso?"
vector_pregunta = modelo.encode([pregunta]).tolist()
resultados = coleccion.query(query_embeddings=vector_pregunta, n_results=2)

print("Pregunta:", pregunta)
print()
for i, doc in enumerate(resultados["documents"][0]):
    distancia = resultados["distances"][0][i]
    categoria = resultados["metadatas"][0][i]["categoria"]
    print(f"{i + 1}. distancia={distancia:.4f}  categoria={categoria}")
    print(f"   {doc}")
    print()


Pregunta: Como me inscribo en el curso?

1. distancia=0.4712  categoria=g4
   Las pruebas de acceso a ciclos formativos se realizan en junio.

2. distancia=0.5182  categoria=g3
   El plan de orientacion ayuda al alumnado a elegir estudios.



## Filtrar por categoría

La cercanía y el filtro no son la misma cosa.

La pregunta es amplia. Sin filtro podrían salir textos de varias categorías. `where` obliga a que la categoría sea `g4` (oferta educativa), igual que un `WHERE` en SQL. Eso no crea otro vector: quita registros y, entre los que quedan, ordena por cercanía.

Si el filtro se aplicara después de pedir los dos más cercanos, se podría perder un texto de oferta que iba en el puesto 3. Por eso va dentro de la misma consulta.


In [6]:
pregunta = "Cuando hay que hacer los tramites?"
vector_pregunta = modelo.encode([pregunta]).tolist()
filtrados = coleccion.query(
    query_embeddings=vector_pregunta,
    n_results=2,
    where={"categoria": "g4"},
)
print("Pregunta:", pregunta)
print("Filtro: categoria = g4")
print()
for doc, meta in zip(filtrados["documents"][0], filtrados["metadatas"][0]):
    print("-", doc, f"({meta['categoria']})")


Pregunta: Cuando hay que hacer los tramites?
Filtro: categoria = g4

- Las pruebas de acceso a ciclos formativos se realizan en junio. (g4)
- Para matricularse hay que presentar la solicitud en secretaria. (g4)


## Memoria y disco

La colección de arriba es un cálculo en memoria. Al cerrar Colab desaparece. Una base de datos tiene que seguir ahí mañana.

`PersistentClient` escribe una carpeta. Se guardan los mismos cinco registros, se cierra el cliente y se abre otro apuntando a la misma carpeta. Si el recuento sigue siendo 5, los vectores han sobrevivido fuera de la variable.

En el proyecto la carpeta no estará en Colab: estará en el servidor. La idea es la misma.


In [7]:
from pathlib import Path

ruta = Path("/content/chroma_ies")
cliente_disco = chromadb.PersistentClient(path=str(ruta))
if "documentacion_ies" in [c.name for c in cliente_disco.list_collections()]:
    cliente_disco.delete_collection("documentacion_ies")

en_disco = cliente_disco.create_collection(
    name="documentacion_ies",
    metadata={"hnsw:space": "cosine"},
)
en_disco.add(ids=ids, documents=frases, embeddings=vectores, metadatas=categorias)

otro = chromadb.PersistentClient(path=str(ruta))
abierta = otro.get_collection("documentacion_ies")
print("Registros al volver a abrir la carpeta:", abierta.count())
print("Carpeta:", ruta)


Registros al volver a abrir la carpeta: 5
Carpeta: /content/chroma_ies


## Ejercicio

1. Añade el texto «El centro dispone de aparcamiento para bicicletas.» con el id `doc6` y la categoría `g5`.
2. Calcula tú el vector con `modelo.encode`. No dejes que ChromaDB lo invente.
3. Pregunta «Donde puedo dejar la bicicleta?» y pide un solo resultado.
4. Imprime el texto y la distancia.

La distancia tiene que ser pequeña, y la categoría tiene que ser `g5`.


In [ ]:
# 1. Calcula el vector de la frase nueva.
# 2. coleccion.add(...) o en_disco.add(...)
# 3. Pregunta y muestra la distancia.
